In [1]:
import numpy as np
import pandas as pd
from scipy.stats import random_correlation
from utils import *
from random import random
from collections import defaultdict

In [2]:
np.random.seed(100)

In [ ]:
N = 10              # supply nodes, demand nodes
NUM_ITRS = 200     # monte carlo sims per world

DEMAND_MU = 100
DEMAND_SIGMA = 25

PROD_CAPACITY = 150

P_FAILS = [0.01, 0.1, 0.2, 0.5]

# Factor-model sweep: one correlation matrix per entry. Fewer factors / smaller
# noise_scale -> more correlated worlds. Pick whichever axis (n_factors or
# noise_scale) gives a clean monotone range of realized mean |correlation|.
FACTOR_SWEEP = [
    # (n_factors, noise_scale)
    (1, 0.2),
    (1, 0.5),
    (2, 0.2),
    (2, 0.5),
    (2, 1.0),
    (3, 0.5),
    (3, 1.0),
    (3, 2.0),
    (5, 1.0),
    (5, 2.0),
]
NUM_WORLDS = len(FACTOR_SWEEP)

In [4]:
# Build our worlds with the factor-model generator and record realized stats.
# We plot against realized mean |correlation|, not against the generator knobs.

rng = np.random.default_rng(100)

corr_mats = []
world_stats = []  # list of dicts with n_factors, noise_scale, mean_abs, std_off, max_off

for world_idx, (n_factors, noise_scale) in enumerate(FACTOR_SWEEP):
    M = generate_corr_matrix_factor(N, n_factors=n_factors,
                                    noise_scale=noise_scale, rng=rng)
    corr_mats.append(M)
    mean_abs, std_off, max_off = corr_summary(M)
    world_stats.append({
        "world": world_idx,
        "n_factors": n_factors,
        "noise_scale": noise_scale,
        "mean_abs_corr": mean_abs,
        "std_corr": std_off,
        "max_corr": max_off,
    })

world_stats_df = pd.DataFrame(world_stats).sort_values("mean_abs_corr")
print(world_stats_df.to_string(index=False))

 world  n_factors  noise_scale  mean_abs_corr  std_corr  max_corr
     2          2          0.2       0.102897  0.132986  0.374499
     0          1          0.2       0.110509  0.118307  0.191286
     1          1          0.5       0.196610  0.278691  0.658548
     5          3          0.5       0.248573  0.287497  0.425127
     3          2          0.5       0.263829  0.314064  0.615815
     8          5          1.0       0.282242  0.336875  0.577741
     6          3          1.0       0.319817  0.375141  0.694511
     9          5          2.0       0.331404  0.362220  0.827614
     4          2          1.0       0.340179  0.374414  0.565361
     7          3          2.0       0.377233  0.437745  0.798722


In [5]:
for i in range(NUM_WORLDS):
    corr = corr_mats[i]
    default_edges = [(j, (j+1) % N) for j in range(N)]
    smart_edges = add_smart_links(corr)  # assuming this returns the cycle edges
    default_sum = sum(corr[a, b] for a, b in default_edges)
    smart_sum = sum(corr[a, b] for a, b in smart_edges)
    print(f"world {i}  mean|ρ|={world_stats[i]['mean_abs_corr']:.3f}  "
          f"default Σρ={default_sum:.3f}  smart Σρ={smart_sum:.3f}  "
          f"gap={default_sum - smart_sum:.3f}")

world 0  mean|ρ|=0.111  default Σρ=0.048  smart Σρ=-0.278  gap=0.326
world 1  mean|ρ|=0.197  default Σρ=-2.048  smart Σρ=-2.693  gap=0.645
world 2  mean|ρ|=0.103  default Σρ=1.057  smart Σρ=-1.070  gap=2.126
world 3  mean|ρ|=0.264  default Σρ=0.344  smart Σρ=-1.998  gap=2.342
world 4  mean|ρ|=0.340  default Σρ=-0.837  smart Σρ=-4.380  gap=3.543
world 5  mean|ρ|=0.249  default Σρ=-0.858  smart Σρ=-3.713  gap=2.855
world 6  mean|ρ|=0.320  default Σρ=0.551  smart Σρ=-3.647  gap=4.198
world 7  mean|ρ|=0.377  default Σρ=-1.071  smart Σρ=-4.715  gap=3.645
world 8  mean|ρ|=0.282  default Σρ=0.049  smart Σρ=-4.394  gap=4.443
world 9  mean|ρ|=0.331  default Σρ=0.117  smart Σρ=-1.354  gap=1.470


In [6]:
from itertools import permutations
# For n=10, there are (n-1)!/2 = 181,440 cycles -- too many to enumerate,
# but you can sample a few thousand.
rng = np.random.default_rng(42)
for i in range(NUM_WORLDS):
    corr = corr_mats[i]
    sums = []
    for _ in range(2000):
        perm = rng.permutation(N)
        edges = [(perm[j], perm[(j+1) % N]) for j in range(N)]
        sums.append(sum(corr[a, b] for a, b in edges))
    smart_sum = sum(corr[a, b] for a, b in add_smart_links(corr))
    pct = (np.array(sums) < smart_sum).mean()
    print(f"world {i}: smart is at the {pct*100:.1f}th percentile of "
          f"random cycles (range: {min(sums):.2f} to {max(sums):.2f})")

world 0: smart is at the 0.0th percentile of random cycles (range: -0.26 to 0.81)
world 1: smart is at the 0.0th percentile of random cycles (range: -2.60 to 2.28)
world 2: smart is at the 0.0th percentile of random cycles (range: -1.03 to 1.31)
world 3: smart is at the 0.0th percentile of random cycles (range: -1.87 to 2.92)
world 4: smart is at the 0.0th percentile of random cycles (range: -4.11 to 3.07)
world 5: smart is at the 0.0th percentile of random cycles (range: -3.24 to 2.09)
world 6: smart is at the 0.0th percentile of random cycles (range: -3.37 to 3.09)
world 7: smart is at the 0.0th percentile of random cycles (range: -4.30 to 4.02)
world 8: smart is at the 0.0th percentile of random cycles (range: -3.40 to 2.44)
world 9: smart is at the 0.0th percentile of random cycles (range: -0.98 to 3.49)


In [7]:
# Build the four chain sets per world, and cache each chain's A_ub so the
# inner loop doesn't rebuild the constraint matrix 5000 times per world.
CONFIG_NAMES = ("no_chain", "default_chain", "smart_chain", "full_flex")

chains_per_world = {name: [] for name in CONFIG_NAMES}
A_ub_per_world = {name: [] for name in CONFIG_NAMES}

for i in range(NUM_WORLDS):
    print(f"Building chains for world i = {i}")

    no_chain = [(j, j) for j in range(N)]

    default_chain = [(j, j) for j in range(N)]
    for j in range(N):
        default_chain.append((j, (j + 1) % N))

    smart_chain = [(j, j) for j in range(N)]
    smart_chain.extend(add_smart_links(corr_mats[i]))

    full_flex = [(j, k) for j in range(N) for k in range(N)]

    world_chains = {
        "no_chain": no_chain,
        "default_chain": default_chain,
        "smart_chain": smart_chain,
        "full_flex": full_flex,
    }

    for name, arcs in world_chains.items():
        chains_per_world[name].append(arcs)
        A_ub_per_world[name].append(build_A_ub(arcs, N, N))

Building chains for world i = 0
Building chains for world i = 1
Building chains for world i = 2
Building chains for world i = 3
Building chains for world i = 4
Building chains for world i = 5
Building chains for world i = 6
Building chains for world i = 7
Building chains for world i = 8
Building chains for world i = 9


In [8]:
# Main simulation loop. Per-replication lost sales are stored for every
# (p, world, rep, config) cell, so downstream analyses (paired diffs, tails)
# have access to the raw data. CRN: one (active, demands) realization per
# (p, world, rep), shared across all four configs.

import time

records = []
t0 = time.time()

for p in P_FAILS:
    print(f"-------- p = {p} --------")
    for i in range(NUM_WORLDS):
        corr = corr_mats[i]
        mean_abs_corr = world_stats[i]["mean_abs_corr"]
        for r in range(NUM_ITRS):
            active = correlated_bernoulli(1 - p, corr)
            capacities = PROD_CAPACITY * np.asarray(active, dtype=float)
            demands = np.clip(
                np.random.normal(DEMAND_MU, DEMAND_SIGMA, size=N),
                0.0, None,
            )
            for name in CONFIG_NAMES:
                ls = solve_transportation_fast(
                    chains_per_world[name][i],
                    capacities,
                    demands,
                    A_ub=A_ub_per_world[name][i],
                )
                records.append((p, i, r, name, ls, mean_abs_corr))
    print(f"  cumulative elapsed: {time.time() - t0:.1f}s")

results_df = pd.DataFrame.from_records(
    records,
    columns=["p", "world", "rep", "config", "lost_sales", "mean_abs_corr"],
)
print(f"\nTotal reps logged: {len(results_df):,}")
results_df.head()

-------- p = 0.01 --------
  cumulative elapsed: 5.4s
-------- p = 0.1 --------
  cumulative elapsed: 11.0s
-------- p = 0.2 --------
  cumulative elapsed: 16.7s
-------- p = 0.5 --------
  cumulative elapsed: 22.3s

Total reps logged: 32,000


,p,world,rep,config,lost_sales,mean_abs_corr
0,0.01,0,0,no_chain,0.000000e+00,0.110509
1,0.01,0,0,default_chain,0.000000e+00,0.110509
2,0.01,0,0,smart_chain,0.000000e+00,0.110509
3,0.01,0,0,full_flex,1.136868e-13,0.110509
4,0.01,0,1,no_chain,0.000000e+00,0.110509


In [9]:
# Marginal means per config (kept for continuity with prior output).
print("--- Marginal mean lost sales per config ---")
for name in CONFIG_NAMES:
    print(f"\n--- {name} ---")
    sub = results_df[results_df["config"] == name]
    for p in P_FAILS:
        avg = sub[sub["p"] == p]["lost_sales"].mean()
        print(f"p = {p}: {avg:.3f}")

--- Marginal mean lost sales per config ---

--- no_chain ---
p = 0.01: 13.350
p = 0.1: 102.998
p = 0.2: 202.935
p = 0.5: 508.262

--- default_chain ---
p = 0.01: 0.646
p = 0.1: 16.352
p = 0.2: 52.524
p = 0.5: 290.071

--- smart_chain ---
p = 0.01: 0.239
p = 0.1: 11.114
p = 0.2: 40.552
p = 0.5: 261.126

--- full_flex ---
p = 0.01: 0.000
p = 0.1: 0.803
p = 0.2: 5.906
p = 0.5: 139.245


In [10]:
# --- Paired differences (CRN) ---
#
# Because every (p, world, rep) cell uses the same capacity and demand
# realization across all four configs, the per-rep paired difference has
# much lower variance than the difference of marginals. This is the right
# statistic for "does correlation-aware design help?"
#
# Bootstrap CI: resample reps *within* each (world, p) cell (stratified),
# never across worlds -- worlds are a separate source of variation handled
# by the per-world breakdown below.

# Pivot to wide: one row per (p, world, rep), one column per config.
wide = (results_df
        .pivot_table(index=["p", "world", "rep", "mean_abs_corr"],
                     columns="config", values="lost_sales")
        .reset_index())

CONTRASTS = {
    "smart - default":       ("smart_chain", "default_chain"),
    "default - full_flex":   ("default_chain", "full_flex"),
    "smart - full_flex":     ("smart_chain", "full_flex"),
}

for label, (a, b) in CONTRASTS.items():
    wide[label] = wide[a] - wide[b]


def paired_bootstrap_ci(df, contrast_col, n_boot=2000, alpha=0.05, seed=0):
    """Stratified paired bootstrap: resample reps within each world,
    recompute the overall mean of contrast_col, repeat. Returns (mean, lo, hi).
    """
    rng = np.random.default_rng(seed)
    worlds = df["world"].unique()
    grouped = {w: df[df["world"] == w][contrast_col].to_numpy()
               for w in worlds}
    observed_mean = float(df[contrast_col].mean())
    boot_means = np.empty(n_boot)
    for k in range(n_boot):
        sample_vals = []
        for w, vals in grouped.items():
            idx = rng.integers(0, len(vals), size=len(vals))
            sample_vals.append(vals[idx])
        boot_means[k] = np.concatenate(sample_vals).mean()
    lo = float(np.percentile(boot_means, 100 * alpha / 2))
    hi = float(np.percentile(boot_means, 100 * (1 - alpha / 2)))
    return observed_mean, lo, hi


contrast_rows = []
for p in P_FAILS:
    sub = wide[wide["p"] == p]
    for label in CONTRASTS:
        mean, lo, hi = paired_bootstrap_ci(sub, label)
        contrast_rows.append({
            "p": p, "contrast": label,
            "mean": mean, "ci_lo": lo, "ci_hi": hi,
        })
contrast_df = pd.DataFrame(contrast_rows)
print("--- Paired differences (mean with stratified-bootstrap 95% CI) ---")
print(contrast_df.to_string(index=False))

--- Paired differences (mean with stratified-bootstrap 95% CI) ---
   p            contrast       mean      ci_lo      ci_hi
0.01     smart - default  -0.407406  -0.797283  -0.109975
0.01 default - full_flex   0.646207   0.279970   1.105549
0.01   smart - full_flex   0.238801   0.065228   0.470187
0.10     smart - default  -5.237848  -6.978554  -3.628613
0.10 default - full_flex  15.548906  13.640683  17.535412
0.10   smart - full_flex  10.311058   8.873833  11.833421
0.20     smart - default -11.971720 -14.629549  -9.414342
0.20 default - full_flex  46.618013  43.419236  49.834931
0.20   smart - full_flex  34.646293  31.848468  37.309862
0.50     smart - default -28.944493 -33.301910 -24.698003
0.50 default - full_flex 150.825904 146.464504 155.318988
0.50   smart - full_flex 121.881411 117.649811 125.966743


In [11]:
# --- Per-world means of each paired contrast ---
# Lets you see whether the smart-vs-default effect is consistent across
# worlds or driven by a few. mean_abs_corr is attached so you can sort.

per_world_rows = []
for p in P_FAILS:
    sub = wide[wide["p"] == p]
    for w, g in sub.groupby("world"):
        row = {"p": p, "world": int(w),
               "mean_abs_corr": float(g["mean_abs_corr"].iloc[0])}
        for label in CONTRASTS:
            row[label] = float(g[label].mean())
        per_world_rows.append(row)
per_world_df = pd.DataFrame(per_world_rows).sort_values(
    ["p", "mean_abs_corr"]).reset_index(drop=True)
print("--- Per-world paired contrasts (sorted by realized mean |corr|) ---")
print(per_world_df.to_string(index=False))

--- Per-world paired contrasts (sorted by realized mean |corr|) ---
   p  world  mean_abs_corr  smart - default  default - full_flex  smart - full_flex
0.01      2       0.102897    -2.158588e+00         2.158588e+00      -1.989520e-14
0.01      0       0.110509     5.396520e-02         1.312958e+00       1.366923e+00
0.01      1       0.196610     2.273737e-15        -2.501110e-14      -2.273737e-14
0.01      5       0.248573    -3.742620e-02         6.136635e-02       2.394015e-02
0.01      3       0.263829     1.633972e-01        -1.818989e-14       1.633972e-01
0.01      8       0.282242     4.792201e-02         1.436945e-01       1.916165e-01
0.01      6       0.319817    -4.018871e-01         4.779357e-01       7.604859e-02
0.01      9       0.331404    -1.852121e-01         1.852121e-01      -1.648459e-14
0.01      4       0.340179     0.000000e+00        -1.080025e-14      -1.080025e-14
0.01      7       0.377233    -1.556231e+00         2.122316e+00       5.660850e-01
0.10    

In [12]:
# --- Tail metrics per configuration (pooled across worlds within each p) ---
# Headline expectation: the smart-vs-naive gap is proportionally larger in
# the tail than in the mean.

def cvar_high(x, alpha=0.95):
    x = np.asarray(x, dtype=float)
    if x.size == 0:
        return float("nan")
    thresh = np.quantile(x, alpha)
    tail = x[x >= thresh]
    return float(tail.mean()) if tail.size else float(thresh)

tail_rows = []
for p in P_FAILS:
    for name in CONFIG_NAMES:
        vals = results_df[(results_df["p"] == p) &
                          (results_df["config"] == name)]["lost_sales"].to_numpy()
        tail_rows.append({
            "p": p, "config": name,
            "mean": float(vals.mean()),
            "p95": float(np.quantile(vals, 0.95)),
            "p99": float(np.quantile(vals, 0.99)),
            "cvar95": cvar_high(vals, 0.95),
        })
tail_df = pd.DataFrame(tail_rows)
print("--- Tail metrics per config (pooled across worlds) ---")
print(tail_df.to_string(index=False))

--- Tail metrics per config (pooled across worlds) ---
   p        config         mean          p95          p99       cvar95
0.01      no_chain 1.335012e+01 9.565860e+01 1.836794e+02 1.572468e+02
0.01 default_chain 6.462072e-01 2.273737e-13 4.547474e-13 7.692942e+00
0.01   smart_chain 2.388011e-01 2.273737e-13 2.296474e-13 2.894559e+00
0.01     full_flex 4.206413e-14 2.273737e-13 2.273737e-13 2.447555e-13
0.10      no_chain 1.029980e+02 3.384026e+02 4.684463e+02 4.252415e+02
0.10 default_chain 1.635221e+01 1.217907e+02 2.386569e+02 1.924131e+02
0.10   smart_chain 1.111436e+01 8.845298e+01 1.875405e+02 1.535878e+02
0.10     full_flex 8.033004e-01 2.273737e-13 4.547474e-13 6.985221e+00
0.20      no_chain 2.029345e+02 4.957951e+02 6.459525e+02 5.863117e+02
0.20 default_chain 5.252356e+01 2.481041e+02 3.913867e+02 3.362162e+02
0.20   smart_chain 4.055184e+01 2.083986e+02 3.796652e+02 3.053091e+02
0.20     full_flex 5.905544e+00 2.273737e-13 2.007810e+02 4.144241e+01
0.50      no_chain 5.0